# 3PTB molecular admission, before searches

This human Python workflow replays the [registered protocol](3ptb_prospective_protocol.md) and [explicit source handoff](3ptb_source_handoff_2026-10-10.md). Molecular operations belong to public MolSysMT APIs; the case-specific admission command only composes them and checks the declared scientific decisions.

One dry rigid receptor: HID57, HIE40/HIE91, six declared disulfides, charged termini. One benzamidinium +1 hypothesis. Two ligand arms share geometry, H, charges and types; only the C1–C torsion changes. No fixed scoring or docking search runs here.

Direct `.pdb.gz` consumption remains unsupported under [MolSysMT #385](https://github.com/uibcdf/molsysmt/issues/385). The caller explicitly restores unchanged plain PDB bytes with the standard `gzip` tool. The command rejects any byte difference before molecular conversion. This is archive handoff, not a new molecular reader.

In [1]:
from pathlib import Path
import gzip
import json
import subprocess
import sys
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'devtools/qualify_3ptb_admission.py').exists())
sys.path.insert(0, str(root))
from devtools.qualify_3ptb_admission import DATA, OUTPUT, admit, registration, sha

payload = OUTPUT.read_bytes()
saved = json.loads(gzip.decompress(payload))
checkpoint = json.loads(OUTPUT.with_name('checkpoint_2026-10-10.json').read_text())
assert sha(payload) == checkpoint['original_producer']['archive_sha256']
assert saved['registration'] == registration()
assert sha(OUTPUT.with_name('producer_2026-10-10.py.txt').read_bytes()) == saved['original_producer']['consumer_driver_sha256']
print(saved['status'], saved['source_counts'])
print('Searches executed:', saved['docking_searches_executed'])

plain_input_admitted {'n_atoms': 1701, 'n_bonds': 21, 'n_groups': 287, 'n_structures': 1}
Searches executed: 0


## Replay public molecular preparation

Use the recorded qualified source/runtime composition (see the admission document), not an unqualified newer provider. `admit` exposes the complete orchestration in the adjacent source: `convert`, `extract`, peptide/template assessment/application, fixed-state H, named charges/types, provider rotatable-bond and rigid-fragment tools, and geometric center. It returns live prepared objects plus detached evidence. The only native Vina calls parse receptor and ligand inputs.

Strict H expansion rejects carrying occupancy/B-factor arrays onto newly generated atoms. The explicit intersection policy retains those original arrays separately and reports their loss from the expanded system. It invents no replacement experimental attributes.

In [2]:
with TemporaryDirectory(prefix='dockingmt-3ptb-notebook-source-') as directory:
    plain = Path(directory) / '3PTB.pdb'
    with plain.open('wb') as output:
        subprocess.run(['gzip', '-dc', str(DATA / '3PTB.pdb.gz')], stdout=output, check=True)
    receptor, ligands, reference, boxes, replay = admit(plain)

for role in ('receptor', 'rigid', 'flexible'):
    assert replay['prepared_inputs'][role]['pdbqt'] == saved['prepared_inputs'][role]['pdbqt']
assert replay['population'] == saved['population']
assert replay['docking_searches_executed'] == 0
pd.DataFrame([
    {'input': role, 'atoms': data['prepared']['n_atoms'],
     'projected_charge_e': data['charge_audit']['total_charge'],
     'named_typing': data['prepared']['metadata']['atom_type_assignment']['rule_version'],
     'native_parser': data['native_vina_parser_admitted'],
     'scientific_assessment': data['preparation_assessment']['assessment']}
    for role, data in saved['prepared_inputs'].items()
])

,input,atoms,projected_charge_e,named_typing,native_parser,scientific_assessment
0,flexible,13,1.0,chemical_environment@1,True,unassessed
1,receptor,2011,6.0,chemical_environment@1,True,unassessed
2,rigid,13,1.0,chemical_environment@1,True,unassessed


In [3]:
pd.DataFrame([
    {'role': role, 'observed_atoms': len(data['expansion']['hydrogen_report']['atom_correspondence']),
     'added_H': data['expansion']['hydrogen_report']['n_added_hydrogens'],
     'dropped_attributes': data['expansion']['hydrogen_report']['dropped_attributes'],
     'strict_rejection': data['expansion']['strict_policy_rejection']['exception'],
     'source_unchanged': data['expansion']['source_unchanged']}
    for role, data in saved.items() if role in ('receptor', 'ligand')
])

,role,observed_atoms,added_H,dropped_attributes,strict_rejection,source_unchanged
0,ligand,9,9,"[b_factor, occupancy]",StructuralInconsistencyError,True
1,receptor,1629,1591,"[b_factor, occupancy]",StructuralInconsistencyError,True


## Inspect the frozen population and source maps

The saved record includes original/expanded snapshots, all provider reports, H parents, full-source selections, prepared/written permutations, ROOT/BRANCH text, original occupancy and B factors, native artifact digests and actual imports. The table below is the planned denominator; every row remains unattempted. Backend box values are prospective adapter projections, not submitted search inputs.

In [4]:
pd.DataFrame([
    {'index': row['index'], 'arm': row['arm'], 'domain': row['domain'],
     'seed': row['protocol']['parameters']['seed'],
     'exhaustiveness': row['protocol']['parameters']['exhaustiveness'],
     'center_A': row['backend_box']['center'], 'size_A': row['backend_box']['size'],
     'status': row['status']}
    for row in saved['population']
])

,index,arm,domain,seed,exhaustiveness,center_A,size_A,status
0,0,rigid,native,7,1,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
1,1,rigid,native,7,8,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
2,2,rigid,native,42,1,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
3,3,rigid,native,42,8,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
4,4,rigid,native,2026,1,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
5,5,rigid,native,2026,8,"[-1.759111, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
6,6,rigid,negative,7,1,"[58.240889, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
7,7,rigid,negative,7,8,"[58.240889, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
8,8,rigid,negative,42,1,"[58.240889, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted
9,9,rigid,negative,42,8,"[58.240889, 14.461, 16.915778]","[20.0, 20.0, 20.0]",not_attempted


In [5]:
maps = {role: data['pdbqt_to_full_source_atom_indices'] for role, data in saved['prepared_inputs'].items() if role != 'receptor'}
print('Written ligand → original full-source map (None = generated H):', maps)
print('Declared disulfide author IDs:', saved['receptor']['disulfide_author_pairs'])
print('Provider fragments:', saved['ligand']['rigid_fragments']['fragment_offsets'])
print('Inside native box:', saved['boxes']['native']['native_heavy_containment'])
print('Inside negative box:', saved['boxes']['negative']['native_heavy_containment'])
print('Source commit:', saved['original_producer']['qualified_provider_commit'])
print('Native artifact SHA-256:', saved['original_producer']['provider_native_artifact']['sha256'])

Written ligand → original full-source map (None = generated H): {'flexible': [1636, 1637, 1638, None, None, None, None, 1630, 1631, 1632, 1633, 1634, 1635], 'rigid': [1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, None, None, None, None]}
Declared disulfide author IDs: [[22, 157], [42, 58], [128, 232], [136, 201], [168, 182], [191, 220]]
Provider fragments: [0, 11, 18]
Inside native box: [True, True, True, True, True, True, True, True, True]
Inside negative box: [False, False, False, False, False, False, False, False, False]
Source commit: 739395d7ea31bec5c3f77cdcb1135ecf7e7c9bac
Native artifact SHA-256: c535c7d2f0e2a92b6ebf8298a60b19e4b5555467b4aa087e9563c6760619827e


## Limits and next decision

Input admission does not establish experimental protonation, independent valence certification, H environmental quality, model suitability, pose recovery, ranking or affinity. The provider's stored-field readiness report explicitly retains unassessed checks; RDKit sanitation and named-field coverage are bounded software evidence. Ring symmetry and amidinium resonance remain limitations for the future source-identity RMSD.

The next scientific operation is the separately recorded 24-row search population. Its execution must consume these exact admitted inputs and retain all attempts, submitted box/bytes, complete returned poses and prespecified positional RMSD. Broader issues #5/#6/#4/#49 stay open. No PharmacophoreMT capability is needed for this question.